# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 38 • Question Answering with Pretrained Transformers

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson develops extractive question answering with Transformer encoders.
It covers context-question tokenization, answer-span alignment, start and end
logits, padding-aware loss, no-answer handling, long-context windows, exact
match, token F1, error analysis, checkpointing, and multilingual considerations.

The notebook contains:

1. a complete offline CPU experiment with a compact Transformer question-answering
   model;
2. optional Hugging Face tokenizer, AutoModelForQuestionAnswering, pipeline, and
   Trainer templates that remain disabled by default.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish extractive and generative question answering;
- combine question and context inputs;
- align character answers with token positions;
- create start and end target indices;
- train a model with start-position and end-position losses;
- mask padded context positions;
- decode valid answer spans;
- represent no-answer examples;
- calculate exact match and token-level F1;
- analyze boundary, no-answer, and retrieval errors;
- explain overflow windows and stride;
- structure a Hugging Face question-answering workflow;
- evaluate Arabic and multilingual QA issues.

## Table of Contents

1. Question Answering Tasks
2. Extractive Question Answering
3. Input Representation
4. Start and End Logits
5. Answer Alignment
6. No-Answer Examples
7. Long Contexts and Overflow Windows
8. Offline QA Dataset
9. Train, Validation, and Test Splits
10. Tokenization and Vocabulary
11. Encoding Questions and Contexts
12. Dataset and Dynamic Padding
13. Transformer QA Model
14. Shape Inspection
15. Loss Function
16. Training Utilities
17. Model Training
18. Learning Curves
19. Span Decoding
20. Exact Match
21. Token-Level F1
22. Validation Evaluation
23. Test Evaluation
24. Qualitative Predictions
25. Error Taxonomy
26. Boundary Errors
27. No-Answer Errors
28. Confidence Analysis
29. Bootstrap Confidence Intervals
30. Checkpoint Saving and Reloading
31. Simulated Sliding Windows
32. Optional Hugging Face Setup
33. Optional Fast-Tokenizer Alignment
34. Optional AutoModel Inference
35. Optional QA Pipeline
36. Optional Trainer Workflow
37. Data Collators and Preprocessing
38. Retrieval Versus Reading
39. Arabic and Multilingual Considerations
40. Reproducibility and Reporting
41. Knowledge Check
42. Exercises
43. Summary and Next Lesson

# 1. Question Answering Tasks

Question answering systems may be:

- extractive;
- generative;
- open-domain;
- closed-domain;
- conversational;
- multiple choice.

In [ ]:
import copy
import importlib.util
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

qa_tasks = pd.DataFrame(
    [
        ("Extractive", "span copied from context"),
        ("Generative", "newly generated answer"),
        ("Open-domain", "retrieval plus answering"),
        ("Conversational", "uses dialogue history"),
    ],
    columns=["QA type", "Output"],
)

qa_tasks

# 2. Extractive Question Answering

Extractive QA predicts a contiguous answer span inside the context.

The model produces one score per token for:

- answer start;
- answer end.

# 3. Input Representation

A common input format is:

```text
[CLS] question [SEP] context [SEP]
```

Segment or token-type embeddings may distinguish question and context tokens.

# 4. Start and End Logits

For sequence length `L`, the model returns:

```text
start_logits: (batch_size, L)
end_logits:   (batch_size, L)
```

In [ ]:
qa_output_shapes = pd.DataFrame(
    [
        ("start_logits", "(batch, sequence_length)"),
        ("end_logits", "(batch, sequence_length)"),
    ],
    columns=["Output", "Shape"],
)

qa_output_shapes

# 5. Answer Alignment

Training data may provide answer character offsets.

The preprocessing pipeline must map those offsets to token indices.

# 6. No-Answer Examples

Some contexts do not contain an answer.

A common convention assigns the no-answer target to the `[CLS]` position.

In [ ]:
NO_ANSWER_POSITION = 0

no_answer_policy = pd.Series(
    {
        "target start": NO_ANSWER_POSITION,
        "target end": NO_ANSWER_POSITION,
        "interpretation": "[CLS] represents no answer",
    }
)

no_answer_policy

# 7. Long Contexts and Overflow Windows

Contexts longer than the model limit are divided into overlapping windows.

Important settings:

- maximum sequence length;
- document stride;
- overflow mapping;
- answer-window selection.

In [ ]:
window_settings = pd.DataFrame(
    [
        ("maximum length", "tokens per model input"),
        ("stride", "overlap between windows"),
        ("overflow mapping", "window-to-example relation"),
        ("offset mapping", "token-to-character relation"),
    ],
    columns=["Setting", "Purpose"],
)

window_settings

# 8. Offline QA Dataset

The synthetic corpus uses person, organization, location, and occupation facts.

In [ ]:
persons = [
    "Alice",
    "Bob",
    "Carol",
    "David",
    "Eman",
    "Farah",
    "George",
    "Hana",
]

organizations = [
    "OpenAI",
    "Google",
    "Microsoft",
    "Amazon",
    "UNESCO",
    "NASA",
    "IBM",
    "Meta",
]

locations = [
    "Cairo",
    "Boston",
    "London",
    "Dubai",
    "Paris",
    "Berlin",
    "Rome",
    "Madrid",
]

occupations = [
    "researcher",
    "engineer",
    "doctor",
    "teacher",
    "analyst",
    "designer",
    "scientist",
    "manager",
]


def make_example(
    index: int,
    no_answer: bool = False,
) -> dict:
    person = persons[index % len(persons)]
    organization = organizations[
        (index * 3) % len(organizations)
    ]
    location = locations[
        (index * 5) % len(locations)
    ]
    occupation = occupations[
        (index * 7) % len(occupations)
    ]

    context = (
        f"{person} works as a {occupation} at "
        f"{organization} in {location}."
    )

    question_type = index % 4

    if question_type == 0:
        question = (
            f"Who works at {organization}?"
        )
        answer = person
    elif question_type == 1:
        question = (
            f"Where does {person} work?"
        )
        answer = location
    elif question_type == 2:
        question = (
            f"What is {person}'s occupation?"
        )
        answer = occupation
    else:
        question = (
            f"Which organization employs {person}?"
        )
        answer = organization

    if no_answer:
        missing_person = persons[
            (index + 3) % len(persons)
        ]
        question = (
            f"What is {missing_person}'s occupation?"
        )
        answer = ""

    return {
        "context": context,
        "question": question,
        "answer": answer,
        "is_answerable": not no_answer,
    }


examples = []

for index in range(160):
    examples.append(
        make_example(
            index,
            no_answer=(
                index % 7 == 0
            ),
        )
    )

dataset = pd.DataFrame(examples)

dataset.head()

In [ ]:
dataset["is_answerable"].value_counts()

# 9. Train, Validation, and Test Splits

In [ ]:
train_frame, test_frame = train_test_split(
    dataset,
    test_size=0.20,
    random_state=42,
    stratify=dataset["is_answerable"],
)

train_frame, validation_frame = train_test_split(
    train_frame,
    test_size=0.20,
    random_state=42,
    stratify=train_frame[
        "is_answerable"
    ],
)

train_frame = train_frame.reset_index(
    drop=True
)
validation_frame = (
    validation_frame.reset_index(
        drop=True
    )
)
test_frame = test_frame.reset_index(
    drop=True
)

pd.Series(
    {
        "training": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    }
)

# 10. Tokenization and Vocabulary

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:['-]\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


special_tokens = [
    "<PAD>",
    "<UNK>",
    "<CLS>",
    "<SEP>",
]

word_counts = Counter(
    token
    for text in pd.concat(
        [
            train_frame["question"],
            train_frame["context"],
        ]
    )
    for token in tokenize(text)
)

vocabulary = (
    special_tokens
    + sorted(word_counts)
)

token2id = {
    token: index
    for index, token
    in enumerate(vocabulary)
}

id2token = {
    index: token
    for token, index
    in token2id.items()
}

PAD_ID = token2id["<PAD>"]
UNK_ID = token2id["<UNK>"]
CLS_ID = token2id["<CLS>"]
SEP_ID = token2id["<SEP>"]

print("Vocabulary size:", len(vocabulary))

# 11. Encoding Questions and Contexts

Answer positions are aligned by matching the answer-token sequence inside the
context-token sequence.

In [ ]:
def find_subsequence(
    sequence: list[str],
    subsequence: list[str],
) -> tuple[int, int] | None:
    if not subsequence:
        return None

    for start in range(
        len(sequence)
        - len(subsequence)
        + 1
    ):
        if (
            sequence[
                start:start
                + len(subsequence)
            ]
            == subsequence
        ):
            return (
                start,
                start
                + len(subsequence)
                - 1,
            )

    return None


def encode_example(
    question: str,
    context: str,
    answer: str,
) -> dict:
    question_tokens = tokenize(
        question
    )
    context_tokens = tokenize(
        context
    )
    answer_tokens = tokenize(
        answer
    )

    tokens = (
        ["<CLS>"]
        + question_tokens
        + ["<SEP>"]
        + context_tokens
        + ["<SEP>"]
    )

    question_end = (
        1 + len(question_tokens)
    )
    context_start = (
        question_end + 1
    )

    token_type_ids = (
        [0] * (context_start)
        + [1] * (
            len(context_tokens) + 1
        )
    )

    input_ids = [
        token2id.get(
            token,
            UNK_ID,
        )
        for token in tokens
    ]

    context_mask = [
        False
    ] * context_start + [
        True
    ] * len(
        context_tokens
    ) + [
        False
    ]

    if not answer_tokens:
        start_position = (
            NO_ANSWER_POSITION
        )
        end_position = (
            NO_ANSWER_POSITION
        )
    else:
        span = find_subsequence(
            context_tokens,
            answer_tokens,
        )

        if span is None:
            raise ValueError(
                f"Answer not found: {answer}"
            )

        start_position = (
            context_start + span[0]
        )
        end_position = (
            context_start + span[1]
        )

    return {
        "tokens": tokens,
        "input_ids": input_ids,
        "token_type_ids": (
            token_type_ids
        ),
        "context_mask": (
            context_mask
        ),
        "start_position": (
            start_position
        ),
        "end_position": (
            end_position
        ),
    }


encoded_example = encode_example(
    train_frame.loc[0, "question"],
    train_frame.loc[0, "context"],
    train_frame.loc[0, "answer"],
)

pd.DataFrame(
    {
        "token": encoded_example[
            "tokens"
        ],
        "type_id": encoded_example[
            "token_type_ids"
        ],
        "context": encoded_example[
            "context_mask"
        ],
    }
)

# 12. Dataset and Dynamic Padding

In [ ]:
class QADataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
    ):
        self.frame = frame.reset_index(
            drop=True
        )

    def __len__(self):
        return len(self.frame)

    def __getitem__(
        self,
        index,
    ):
        row = self.frame.iloc[index]

        encoded = encode_example(
            row["question"],
            row["context"],
            row["answer"],
        )

        return {
            "input_ids": torch.tensor(
                encoded["input_ids"],
                dtype=torch.long,
            ),
            "token_type_ids": torch.tensor(
                encoded[
                    "token_type_ids"
                ],
                dtype=torch.long,
            ),
            "context_mask": torch.tensor(
                encoded[
                    "context_mask"
                ],
                dtype=torch.bool,
            ),
            "start_position": torch.tensor(
                encoded[
                    "start_position"
                ],
                dtype=torch.long,
            ),
            "end_position": torch.tensor(
                encoded[
                    "end_position"
                ],
                dtype=torch.long,
            ),
            "tokens": encoded["tokens"],
            "question": row["question"],
            "context": row["context"],
            "answer": row["answer"],
        }


def collate_qa_batch(
    batch,
):
    maximum_length = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            maximum_length,
        ),
        PAD_ID,
        dtype=torch.long,
    )

    token_type_ids = torch.zeros(
        (
            len(batch),
            maximum_length,
        ),
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (
            len(batch),
            maximum_length,
        ),
        dtype=torch.long,
    )

    context_mask = torch.zeros(
        (
            len(batch),
            maximum_length,
        ),
        dtype=torch.bool,
    )

    for row, item in enumerate(batch):
        length = len(
            item["input_ids"]
        )

        input_ids[
            row,
            :length,
        ] = item["input_ids"]

        token_type_ids[
            row,
            :length,
        ] = item[
            "token_type_ids"
        ]

        attention_mask[
            row,
            :length,
        ] = 1

        context_mask[
            row,
            :length,
        ] = item[
            "context_mask"
        ]

    return {
        "input_ids": input_ids,
        "token_type_ids": (
            token_type_ids
        ),
        "attention_mask": (
            attention_mask
        ),
        "padding_mask": (
            attention_mask == 0
        ),
        "context_mask": (
            context_mask
        ),
        "start_positions": (
            torch.stack(
                [
                    item[
                        "start_position"
                    ]
                    for item in batch
                ]
            )
        ),
        "end_positions": (
            torch.stack(
                [
                    item[
                        "end_position"
                    ]
                    for item in batch
                ]
            )
        ),
        "tokens": [
            item["tokens"]
            for item in batch
        ],
        "questions": [
            item["question"]
            for item in batch
        ],
        "contexts": [
            item["context"]
            for item in batch
        ],
        "answers": [
            item["answer"]
            for item in batch
        ],
    }


train_dataset = QADataset(
    train_frame
)
validation_dataset = QADataset(
    validation_frame
)
test_dataset = QADataset(
    test_frame
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_qa_batch,
    generator=torch.Generator().manual_seed(
        42
    ),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_qa_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_qa_batch,
)

sample_batch = next(
    iter(train_loader)
)

print(
    sample_batch[
        "input_ids"
    ].shape
)

# 13. Transformer QA Model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        model_dimension: int,
        maximum_length: int = 128,
    ):
        super().__init__()

        encoding = torch.zeros(
            maximum_length,
            model_dimension,
        )

        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (
                -math.log(10000.0)
                / model_dimension
            )
        )

        encoding[:, 0::2] = torch.sin(
            positions * rates
        )
        encoding[:, 1::2] = torch.cos(
            positions * rates
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        return (
            embeddings
            + self.encoding[
                :,
                :embeddings.size(1),
                :,
            ]
        )


class TransformerQAModel(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 48,
        head_count: int = 4,
        layer_count: int = 2,
        feed_forward_dimension: int = 96,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.model_dimension = (
            model_dimension
        )

        self.token_embedding = (
            nn.Embedding(
                vocabulary_size,
                model_dimension,
                padding_idx=PAD_ID,
            )
        )

        self.segment_embedding = (
            nn.Embedding(
                2,
                model_dimension,
            )
        )

        self.position = (
            PositionalEncoding(
                model_dimension
            )
        )

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=(
                feed_forward_dimension
            ),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = (
            nn.TransformerEncoder(
                layer,
                num_layers=layer_count,
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.qa_outputs = nn.Linear(
            model_dimension,
            2,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        token_type_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        embeddings = (
            self.token_embedding(
                input_ids
            )
            + self.segment_embedding(
                token_type_ids
            )
        )

        embeddings = (
            embeddings
            * math.sqrt(
                self.model_dimension
            )
        )

        hidden_states = self.encoder(
            self.position(
                embeddings
            ),
            src_key_padding_mask=(
                padding_mask
            ),
        )

        logits = self.qa_outputs(
            self.dropout(
                hidden_states
            )
        )

        start_logits = logits[
            :,
            :,
            0,
        ]

        end_logits = logits[
            :,
            :,
            1,
        ]

        return {
            "start_logits": (
                start_logits
            ),
            "end_logits": (
                end_logits
            ),
            "last_hidden_state": (
                hidden_states
            ),
        }


DEVICE = torch.device("cpu")

torch.manual_seed(42)

model = TransformerQAModel(
    vocabulary_size=len(vocabulary)
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter
        in model.parameters()
    ),
)

# 14. Shape Inspection

In [ ]:
with torch.no_grad():
    shape_output = model(
        sample_batch[
            "input_ids"
        ].to(DEVICE),
        sample_batch[
            "token_type_ids"
        ].to(DEVICE),
        sample_batch[
            "padding_mask"
        ].to(DEVICE),
    )

print(
    "Start logits:",
    shape_output[
        "start_logits"
    ].shape,
)
print(
    "End logits:",
    shape_output[
        "end_logits"
    ].shape,
)

# 15. Loss Function

The total loss is the mean of start-position and end-position cross-entropy.

In [ ]:
position_loss = nn.CrossEntropyLoss()


def qa_loss(
    start_logits: torch.Tensor,
    end_logits: torch.Tensor,
    start_positions: torch.Tensor,
    end_positions: torch.Tensor,
) -> torch.Tensor:
    start_loss = position_loss(
        start_logits,
        start_positions,
    )

    end_loss = position_loss(
        end_logits,
        end_positions,
    )

    return (
        start_loss + end_loss
    ) / 2.0

# 16. Training Utilities

In [ ]:
def set_seed(
    seed: int = 42,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def mask_non_answer_positions(
    logits: torch.Tensor,
    context_mask: torch.Tensor,
) -> torch.Tensor:
    allowed = context_mask.clone()
    allowed[:, 0] = True

    return logits.masked_fill(
        ~allowed,
        -1e9,
    )


def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
) -> float:
    model.eval()

    losses = []

    with torch.no_grad():
        for batch in loader:
            output = model(
                batch[
                    "input_ids"
                ].to(DEVICE),
                batch[
                    "token_type_ids"
                ].to(DEVICE),
                batch[
                    "padding_mask"
                ].to(DEVICE),
            )

            start_logits = (
                mask_non_answer_positions(
                    output[
                        "start_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            end_logits = (
                mask_non_answer_positions(
                    output[
                        "end_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            loss = qa_loss(
                start_logits,
                end_logits,
                batch[
                    "start_positions"
                ].to(DEVICE),
                batch[
                    "end_positions"
                ].to(DEVICE),
            )

            losses.append(
                float(loss.item())
            )

    return float(
        np.mean(losses)
    )

# 17. Model Training

In [ ]:
def train_model(
    model: nn.Module,
    epochs: int = 50,
    learning_rate: float = 0.003,
    patience: int = 9,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float(
        "inf"
    )
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            optimizer.zero_grad()

            output = model(
                batch[
                    "input_ids"
                ].to(DEVICE),
                batch[
                    "token_type_ids"
                ].to(DEVICE),
                batch[
                    "padding_mask"
                ].to(DEVICE),
            )

            start_logits = (
                mask_non_answer_positions(
                    output[
                        "start_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            end_logits = (
                mask_non_answer_positions(
                    output[
                        "end_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            loss = qa_loss(
                start_logits,
                end_logits,
                batch[
                    "start_positions"
                ].to(DEVICE),
                batch[
                    "end_positions"
                ].to(DEVICE),
            )

            loss.backward()

            gradient_norm = (
                clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )
            )

            optimizer.step()

            training_losses.append(
                float(loss.item())
            )

            gradient_norms.append(
                float(
                    gradient_norm
                )
            )

        validation_loss = (
            evaluate_loss(
                model,
                validation_loader,
            )
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(
                    np.mean(
                        training_losses
                    )
                ),
                "validation_loss": (
                    validation_loss
                ),
                "gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

        if (
            validation_loss
            < best_validation_loss
            - 1e-5
        ):
            best_validation_loss = (
                validation_loss
            )
            best_state = copy.deepcopy(
                model.state_dict()
            )
            without_improvement = 0
        else:
            without_improvement += 1

        if (
            without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )


set_seed(42)

trained_model, training_history = (
    train_model(model)
)

print(
    "Epochs completed:",
    len(training_history),
)
print(
    "Best validation loss:",
    round(
        training_history[
            "validation_loss"
        ].min(),
        4,
    ),
)

# 18. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history[
        "training_loss"
    ],
    label="Training loss",
)
plt.plot(
    training_history["epoch"],
    training_history[
        "validation_loss"
    ],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Question Answering Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 19. Span Decoding

Decoding searches for a valid start-end pair:

- both positions must belong to the context, or both must be `[CLS]`;
- end must not precede start;
- span length should remain bounded.

In [ ]:
def decode_best_span(
    start_logits: torch.Tensor,
    end_logits: torch.Tensor,
    tokens: list[str],
    context_mask: torch.Tensor,
    maximum_answer_length: int = 6,
) -> dict:
    start_scores = (
        start_logits.detach()
        .cpu()
        .numpy()
    )

    end_scores = (
        end_logits.detach()
        .cpu()
        .numpy()
    )

    context_flags = (
        context_mask.detach()
        .cpu()
        .numpy()
    )

    no_answer_score = (
        start_scores[0]
        + end_scores[0]
    )

    best_score = (
        no_answer_score
    )
    best_start = 0
    best_end = 0

    for start in range(
        1,
        len(tokens),
    ):
        if not context_flags[start]:
            continue

        maximum_end = min(
            len(tokens) - 1,
            start
            + maximum_answer_length
            - 1,
        )

        for end in range(
            start,
            maximum_end + 1,
        ):
            if not context_flags[end]:
                continue

            score = (
                start_scores[start]
                + end_scores[end]
            )

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    if best_start == 0:
        answer = ""
    else:
        answer = " ".join(
            tokens[
                best_start:
                best_end + 1
            ]
        )

    return {
        "answer": answer,
        "start": best_start,
        "end": best_end,
        "score": float(
            best_score
        ),
        "no_answer_score": float(
            no_answer_score
        ),
    }

# 20. Exact Match

Exact match equals 1 when the normalized prediction exactly matches the normalized
reference.

In [ ]:
def normalize_answer(
    text: str,
) -> str:
    tokens = tokenize(text)
    return " ".join(tokens)


def exact_match(
    reference: str,
    prediction: str,
) -> float:
    return float(
        normalize_answer(reference)
        == normalize_answer(
            prediction
        )
    )


exact_match(
    "Cairo",
    "cairo",
)

# 21. Token-Level F1

In [ ]:
def token_f1(
    reference: str,
    prediction: str,
) -> float:
    reference_tokens = tokenize(
        reference
    )
    prediction_tokens = tokenize(
        prediction
    )

    if (
        not reference_tokens
        and not prediction_tokens
    ):
        return 1.0

    if (
        not reference_tokens
        or not prediction_tokens
    ):
        return 0.0

    reference_counts = Counter(
        reference_tokens
    )
    prediction_counts = Counter(
        prediction_tokens
    )

    overlap = sum(
        (
            reference_counts
            & prediction_counts
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap
        / len(
            prediction_tokens
        )
    )

    recall = (
        overlap
        / len(
            reference_tokens
        )
    )

    return (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

# 22. Validation Evaluation

In [ ]:
def evaluate_predictions(
    model: nn.Module,
    loader: DataLoader,
) -> pd.DataFrame:
    model.eval()

    rows = []

    with torch.no_grad():
        for batch in loader:
            output = model(
                batch[
                    "input_ids"
                ].to(DEVICE),
                batch[
                    "token_type_ids"
                ].to(DEVICE),
                batch[
                    "padding_mask"
                ].to(DEVICE),
            )

            start_logits = (
                mask_non_answer_positions(
                    output[
                        "start_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            end_logits = (
                mask_non_answer_positions(
                    output[
                        "end_logits"
                    ],
                    batch[
                        "context_mask"
                    ].to(DEVICE),
                )
            )

            for row_index, tokens in enumerate(
                batch["tokens"]
            ):
                decoded = (
                    decode_best_span(
                        start_logits[
                            row_index
                        ],
                        end_logits[
                            row_index
                        ],
                        tokens,
                        batch[
                            "context_mask"
                        ][
                            row_index
                        ],
                    )
                )

                reference = batch[
                    "answers"
                ][row_index]

                prediction = (
                    decoded[
                        "answer"
                    ]
                )

                rows.append(
                    {
                        "question": batch[
                            "questions"
                        ][row_index],
                        "context": batch[
                            "contexts"
                        ][row_index],
                        "reference": (
                            reference
                        ),
                        "prediction": (
                            prediction
                        ),
                        "exact_match": (
                            exact_match(
                                reference,
                                prediction,
                            )
                        ),
                        "token_f1": (
                            token_f1(
                                reference,
                                prediction,
                            )
                        ),
                        "is_answerable": (
                            bool(
                                normalize_answer(
                                    reference
                                )
                            )
                        ),
                        "score_margin": (
                            decoded[
                                "score"
                            ]
                            - decoded[
                                "no_answer_score"
                            ]
                        ),
                    }
                )

    return pd.DataFrame(rows)


validation_results = (
    evaluate_predictions(
        trained_model,
        validation_loader,
    )
)

validation_results[
    [
        "exact_match",
        "token_f1",
    ]
].mean()

# 23. Test Evaluation

In [ ]:
test_results = evaluate_predictions(
    trained_model,
    test_loader,
)

pd.Series(
    {
        "Exact match": (
            test_results[
                "exact_match"
            ].mean()
        ),
        "Token F1": (
            test_results[
                "token_f1"
            ].mean()
        ),
        "Answerable EM": (
            test_results.loc[
                test_results[
                    "is_answerable"
                ],
                "exact_match",
            ].mean()
        ),
        "No-answer EM": (
            test_results.loc[
                ~test_results[
                    "is_answerable"
                ],
                "exact_match",
            ].mean()
        ),
    }
)

# 24. Qualitative Predictions

In [ ]:
test_results[
    [
        "question",
        "context",
        "reference",
        "prediction",
        "exact_match",
        "token_f1",
    ]
].head(12)

# 25. Error Taxonomy

In [ ]:
def categorize_qa_error(
    reference: str,
    prediction: str,
) -> str:
    reference_normalized = (
        normalize_answer(
            reference
        )
    )
    prediction_normalized = (
        normalize_answer(
            prediction
        )
    )

    if (
        reference_normalized
        == prediction_normalized
    ):
        return "correct"

    if (
        not reference_normalized
        and prediction_normalized
    ):
        return (
            "false positive answer"
        )

    if (
        reference_normalized
        and not prediction_normalized
    ):
        return (
            "missed answer"
        )

    if (
        token_f1(
            reference,
            prediction,
        )
        > 0.0
    ):
        return (
            "boundary overlap"
        )

    return "wrong span"


test_results[
    "error_type"
] = [
    categorize_qa_error(
        reference,
        prediction,
    )
    for reference, prediction
    in zip(
        test_results[
            "reference"
        ],
        test_results[
            "prediction"
        ],
    )
]

test_results[
    "error_type"
].value_counts()

# 26. Boundary Errors

Boundary errors contain some correct answer tokens but include too many or too few
tokens.

# 27. No-Answer Errors

No-answer evaluation distinguishes:

- false positive answer;
- missed answer;
- correct no-answer prediction.

# 28. Confidence Analysis

In [ ]:
test_results.groupby(
    "error_type"
)["score_margin"].describe()

The score margin is only a model score diagnostic. It is not automatically a
calibrated probability.

# 29. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_mean_interval(
    values,
    repetitions: int = 1000,
    seed: int = 42,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    generator = np.random.default_rng(
        seed
    )

    means = []

    for _ in range(repetitions):
        indices = generator.integers(
            0,
            len(values),
            size=len(values),
        )

        means.append(
            float(
                values[
                    indices
                ].mean()
            )
        )

    return {
        "mean": float(
            values.mean()
        ),
        "ci_lower": float(
            np.quantile(
                means,
                0.025,
            )
        ),
        "ci_upper": float(
            np.quantile(
                means,
                0.975,
            )
        ),
    }


pd.DataFrame(
    [
        {
            "metric": "Exact match",
            **bootstrap_mean_interval(
                test_results[
                    "exact_match"
                ]
            ),
        },
        {
            "metric": "Token F1",
            **bootstrap_mean_interval(
                test_results[
                    "token_f1"
                ]
            ),
        },
    ]
)

# 30. Checkpoint Saving and Reloading

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = (
        Path(directory)
        / "qa_model.pt"
    )

    torch.save(
        {
            "model_state_dict": (
                trained_model.state_dict()
            ),
            "vocabulary": vocabulary,
            "token2id": token2id,
        },
        checkpoint_path,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model = TransformerQAModel(
        vocabulary_size=len(
            checkpoint[
                "vocabulary"
            ]
        )
    ).to(DEVICE)

    reloaded_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    reload_results = (
        evaluate_predictions(
            reloaded_model,
            test_loader,
        )
    )

print(
    "Reloaded exact match:",
    round(
        reload_results[
            "exact_match"
        ].mean(),
        3,
    ),
)

# 31. Simulated Sliding Windows

In [ ]:
def create_windows(
    tokens: list[str],
    maximum_context_tokens: int,
    stride: int,
) -> list[dict]:
    windows = []

    start = 0

    while start < len(tokens):
        end = min(
            len(tokens),
            start
            + maximum_context_tokens,
        )

        windows.append(
            {
                "start": start,
                "end": end,
                "tokens": tokens[
                    start:end
                ],
            }
        )

        if end == len(tokens):
            break

        start += (
            maximum_context_tokens
            - stride
        )

    return windows


long_context_tokens = tokenize(
    "Alice visited Cairo before joining "
    "OpenAI in Boston and later speaking "
    "at a conference in London."
)

pd.DataFrame(
    create_windows(
        long_context_tokens,
        maximum_context_tokens=7,
        stride=2,
    )
)

Overlap prevents answers near window boundaries from being split across windows.

# 32. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "transformers"
    )
    is not None
)

DATASETS_AVAILABLE = (
    importlib.util.find_spec(
        "datasets"
    )
    is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True

MODEL_ID = (
    "distilbert/"
    "distilbert-base-uncased"
)

pd.Series(
    {
        "transformers installed": (
            TRANSFORMERS_AVAILABLE
        ),
        "datasets installed": (
            DATASETS_AVAILABLE
        ),
        "run demos": (
            RUN_HUGGING_FACE_DEMOS
        ),
        "local files only": (
            USE_LOCAL_FILES_ONLY
        ),
        "model ID": MODEL_ID,
    }
)

# 33. Optional Fast-Tokenizer Alignment

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import (
        AutoTokenizer,
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_ID,
            use_fast=True,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
    )

    question = (
        "Where does Alice work?"
    )
    context = (
        "Alice works at OpenAI in Cairo."
    )

    encoded = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=64,
        stride=16,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt",
    )

    print(
        "input_ids:",
        encoded[
            "input_ids"
        ].shape,
    )
    print(
        "offset_mapping:",
        encoded[
            "offset_mapping"
        ].shape,
    )
else:
    print(
        "Optional fast-tokenizer alignment skipped."
    )

# 34. Optional AutoModel Inference

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import (
        AutoModelForQuestionAnswering,
    )

    hf_model = (
        AutoModelForQuestionAnswering
        .from_pretrained(
            MODEL_ID,
            local_files_only=(
                USE_LOCAL_FILES_ONLY
            ),
        )
        .to("cpu")
    )

    model_inputs = {
        key: value
        for key, value
        in encoded.items()
        if key not in {
            "offset_mapping",
            "overflow_to_sample_mapping",
        }
    }

    with torch.no_grad():
        hf_output = hf_model(
            **model_inputs
        )

    print(
        "Start logits:",
        hf_output[
            "start_logits"
        ].shape,
    )
    print(
        "End logits:",
        hf_output[
            "end_logits"
        ].shape,
    )
else:
    print(
        "Optional AutoModel inference skipped."
    )

# 35. Optional QA Pipeline

In [ ]:
if (
    TRANSFORMERS_AVAILABLE
    and RUN_HUGGING_FACE_DEMOS
):
    from transformers import pipeline

    qa_pipeline = pipeline(
        task="question-answering",
        model=MODEL_ID,
        tokenizer=MODEL_ID,
        device=-1,
        model_kwargs={
            "local_files_only": (
                USE_LOCAL_FILES_ONLY
            )
        },
    )

    result = qa_pipeline(
        question=(
            "Where does Alice work?"
        ),
        context=(
            "Alice works at OpenAI in Cairo."
        ),
    )

    print(result)
else:
    print(
        "Optional QA pipeline skipped."
    )

# 36. Optional Trainer Workflow

The following template is version-sensitive and is not executed.

In [ ]:
trainer_template = '''
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_ID
)

data_collator = DefaultDataCollator()

arguments = TrainingArguments(
    output_dir="checkpoints/question-answering",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=train_features,
    eval_dataset=validation_features,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()
'''

print(trainer_template)

Exact Trainer arguments should be verified against the installed Transformers
version.

# 37. Data Collators and Preprocessing

QA preprocessing typically creates:

- input IDs;
- attention masks;
- optional token-type IDs;
- start positions;
- end positions;
- overflow mappings;
- offset mappings.

# 38. Retrieval Versus Reading

Open-domain QA has two major stages:

1. retrieval selects relevant passages;
2. the reader extracts or generates an answer.

Reader accuracy cannot compensate for missing retrieval evidence.

In [ ]:
open_domain_pipeline = pd.DataFrame(
    [
        (1, "Question"),
        (2, "Retriever"),
        (3, "Candidate passages"),
        (4, "Reader"),
        (5, "Answer and confidence"),
    ],
    columns=["Stage", "Component"],
)

open_domain_pipeline

# 39. Arabic and Multilingual Considerations

Arabic QA is affected by:

- attached clitics;
- optional tashkeel;
- rich morphology;
- orthographic variation;
- answer-boundary fragmentation;
- MSA and dialect variation;
- right-to-left display;
- multilingual retrieval quality.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "أَيْنَ تَعْمَلُ إِيمَانُ؟",
            "تَعْمَلُ إِيمَانُ فِي جَامِعَةِ عَيْنِ شَمْسٍ.",
            "جَامِعَةِ عَيْنِ شَمْسٍ",
        ),
        (
            "مَنْ يَعْمَلُ فِي الْقَاهِرَةِ؟",
            "يَعْمَلُ عَلِيٌّ فِي الْقَاهِرَةِ.",
            "عَلِيٌّ",
        ),
    ],
    columns=[
        "Question",
        "Context",
        "Answer span",
    ],
)

arabic_examples

For fully vocalized Arabic QA, tashkeel must remain in the question, context,
offsets, token alignment, answer span, and evaluation whenever it is part of the
task definition.

Any normalization step that changes characters must update answer offsets.

In [ ]:
arabic_qa_checks = pd.DataFrame(
    [
        ("Tashkeel", "preserve offsets consistently"),
        ("Clitics", "inspect answer-boundary splits"),
        ("Normalization", "recalculate character offsets"),
        ("Variety", "separate MSA and dialect analysis"),
        ("Retrieval", "evaluate Arabic passage recall"),
    ],
    columns=["Check", "Action"],
)

arabic_qa_checks

# 40. Reproducibility and Reporting

Report:

- dataset and split;
- answerability distribution;
- tokenizer and revision;
- maximum length and stride;
- character-to-token alignment method;
- no-answer convention;
- model and checkpoint revision;
- learning rate;
- batch size;
- checkpoint-selection criterion;
- exact match and token F1;
- answerable and no-answer results;
- confidence intervals;
- random seeds;
- software versions;
- hardware;
- documented limitations.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        "training examples": len(
            train_frame
        ),
        "validation examples": len(
            validation_frame
        ),
        "test examples": len(
            test_frame
        ),
        "no-answer position": (
            NO_ANSWER_POSITION
        ),
        "device": str(DEVICE),
        "seed": 42,
        "python": (
            platform.python_version()
        ),
        "torch": torch.__version__,
        "transformers installed": (
            TRANSFORMERS_AVAILABLE
        ),
        "optional demos enabled": (
            RUN_HUGGING_FACE_DEMOS
        ),
    },
    name="Lesson 38 experiment",
)

reproducibility_metadata

# 41. Knowledge Check

1. What is extractive question answering?
2. What do start and end logits represent?
3. Why must answer character offsets be aligned to tokens?
4. How can `[CLS]` represent no answer?
5. Why are non-context tokens masked during decoding?
6. Why must end be greater than or equal to start?
7. What does exact match measure?
8. What does token F1 measure?
9. What is a boundary error?
10. Why are overlapping windows used?
11. What does document stride control?
12. Why is offset mapping important?
13. How do retrieval and reading differ?
14. Why can Arabic normalization break answer offsets?
15. Why should answerable and no-answer results be reported separately?

# 42. Exercises

## Exercise 1 — Character Alignment

Convert answer character offsets to token start and end positions.

## Exercise 2 — No-Answer Threshold

Tune a no-answer threshold on validation data.

## Exercise 3 — Sliding Windows

Create overlapping context windows and map predictions back to the source example.

## Exercise 4 — Maximum Span Length

Compare several maximum answer lengths.

## Exercise 5 — Error Analysis

Categorize missed, false-positive, boundary, and wrong-span errors.

## Exercise 6 — Hugging Face Fine-Tuning

Fine-tune AutoModelForQuestionAnswering on a public QA dataset.

## Exercise 7 — SQuAD Metrics

Use an established exact-match and F1 implementation.

## Exercise 8 — Retrieval

Add a simple passage retriever before the reader.

## Exercise 9 — Arabic QA

Build a fully vocalized MSA extractive QA dataset.

## Exercise 10 — Model Card

Document model scope, limitations, and evaluation.

## Challenge Exercises

1. Add SQuAD 2.0-style no-answer scoring.
2. Compare extractive and generative QA.
3. Add passage reranking.
4. Evaluate multilingual and Arabic-specific readers.
5. Build a retrieval-augmented QA pipeline with evidence citations.

# 43. Summary and Next Lesson

In this lesson:

- extractive question answering was formulated as start and end prediction;
- questions and contexts were combined into one Transformer input;
- answer spans were aligned with context tokens;
- no-answer examples used the `[CLS]` position;
- padded and non-context positions were masked;
- a complete CPU-only Transformer QA model was trained;
- valid answer spans were decoded with length constraints;
- exact match and token F1 were calculated;
- answerable and no-answer performance were separated;
- boundary, missed-answer, false-positive, and wrong-span errors were analyzed;
- bootstrap confidence intervals quantified evaluation uncertainty;
- sliding windows and overflow processing were introduced;
- optional Hugging Face tokenizer, AutoModel, pipeline, and Trainer workflows
  were provided;
- Arabic morphology, tashkeel, normalization, offsets, and multilingual retrieval
  were integrated into the QA workflow.

## Next Lesson

**Lesson 39: Text Generation with Pretrained Language Models** introduces
pretrained causal language models, prompt construction, greedy decoding,
sampling, temperature, top-k, top-p, repetition control, stopping criteria,
evaluation, and responsible generation.

# References

- Hugging Face Transformers documentation: question answering, fast tokenizers,
  overflow windows, pipelines, and Trainer.
- Rajpurkar, P. et al. SQuAD.
- Devlin, J. et al. BERT.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.